In [13]:
import pandas as pd
import torch
import lightning as pl
import wandb
from torchmetrics.classification import Accuracy, F1Score
from lightning.pytorch.loggers import WandbLogger

from configs import configs

## Завантаження даних

In [14]:
x_train = pd.read_csv("data/processed_data/x_train.csv").astype(float)
y_train = pd.read_csv("data/processed_data/y_train.csv").astype(float)

x_val = pd.read_csv("data/processed_data/x_val.csv").astype(float)
y_val = pd.read_csv("data/processed_data/y_val.csv").astype(float)

In [15]:
class TitanicDataset(torch.utils.data.Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return torch.tensor(self.x.iloc[idx].values, dtype=torch.float32), torch.tensor(self.y.iloc[idx].values, dtype=torch.float32)

In [16]:
train_dataset = TitanicDataset(x_train, y_train)
val_dataset = TitanicDataset(x_val, y_val)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=configs["batch_size"], shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=configs["batch_size"], shuffle=False)

## Створення моделі

In [21]:
class CustomModel(pl.LightningModule):
    def __init__(self, lr):
        super().__init__()
        self.model = torch.nn.Sequential(
            torch.nn.Linear(19, 64),
            torch.nn.ReLU(),
            torch.nn.Linear(64, 1),
            torch.nn.Sigmoid()
        )
        self.train_accuracy = Accuracy(task="binary")
        self.val_accuracy = Accuracy(task="binary")
        self.train_f1 = F1Score(num_classes=2, task="binary")
        self.val_f1 = F1Score(num_classes=2, task="binary")
        self.loss_fn = torch.nn.BCELoss()
        self.lr = lr

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)
        self.train_accuracy.update(y_hat, y)
        self.train_f1.update(y_hat, y)
        #self.log("train/loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)
        self.val_accuracy.update(y_hat, y)
        self.val_f1.update(y_hat, y)
        #self.log("val/loss", loss)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)
    
    #def on_train_epoch_end(self):    
        #self.log("train_acc_epoch", self.train_accuracy.compute())
        #self.log("train_f1_epoch", self.train_f1.compute())
    
    #def on_validation_epoch_end(self):
        #self.log("val_acc_epoch", self.val_accuracy.compute())
        #self.log("val_f1_epoch", self.val_f1.compute())

In [23]:
model = CustomModel(lr=configs["lr"])

## Тренування моделі

In [24]:
trainer = pl.Trainer(max_epochs=configs["epochs"], accelerator="gpu", devices=[0])

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [25]:
trainer.fit(model, train_loader, val_loader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name           | Type           | Params
--------------------------------------------------
0 | model          | Sequential     | 1.3 K 
1 | train_accuracy | BinaryAccuracy | 0     
2 | val_accuracy   | BinaryAccuracy | 0     
3 | train_f1       | BinaryF1Score  | 0     
4 | val_f1         | BinaryF1Score  | 0     
5 | loss_fn        | BCELoss        | 0     
--------------------------------------------------
1.3 K     Trainable params
0         Non-trainable params
1.3 K     Total params
0.005     Total estimated model params size (MB)


Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

c:\Users\avhrs\Developer\python-learn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:441: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x9 and 19x64)